# 1. What you'll learn

        - turn token counts into smoothed class likelihoods
- trace token evidence for one prediction
- break unigrams with order-dependent phrases

        **The one question this notebook answers:** How can word counts route messages, and where does the independence assumption fail?

# 2. Setup

This lesson keeps naive bayes for text small enough to inspect end to end. Fixed seeds and one shared split make every comparison reproducible.

The seed is fixed at 0 for data, splits, optimization, and sampling.
Versions are printed so this execution remains an auditable experiment,
not a result detached from the software that produced it.

In [ ]:
from pathlib import Path
import platform
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="colorblind")

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Seed: {SEED}")

# 3. The data

Three hundred short support messages belong to billing, technical, or account queues. The target is the queue; generic filler words make the task less artificial.

**Small example:** If 'refund' occurs 4 times among 20 billing tokens, add-one smoothing with a 10-word vocabulary estimates (4+1)/(20+10)=1/6 instead of risking zero.

In [ ]:
rng=np.random.default_rng(SEED)
templates={"billing":["refund my payment","invoice charged twice","cancel subscription fee","card payment disputed"],"technical":["app crashes on launch","server error after update","page loads very slowly","cannot install software"],"account":["reset my password","unlock my account","change login email","enable two factor access"]}
fillers=["please","today","urgent","help","customer"];texts=[];labels=[]
for label,items in templates.items():
    for i in range(100):
        texts.append(items[i%4]+" "+" ".join(rng.choice(fillers,size=rng.integers(0,3),replace=False)));labels.append(label)
frame=pd.DataFrame({"text":texts,"queue":labels});print("Shape:",frame.shape,"balance:",frame.queue.value_counts().to_dict());print(frame.sample(5,random_state=SEED))

# 4. Explore

The three views establish the target, the useful structure, and the specific data property that will govern the model's success or failure.

The next code produces three purposeful panels. Read the written
takeaways underneath before moving on; the plots are evidence for later
modeling choices, not decoration.

In [ ]:
vec=CountVectorizer().fit(texts);M=vec.transform(texts);names=np.array(vec.get_feature_names_out());counts=np.asarray(M.sum(axis=0)).ravel();top=np.argsort(counts)[-10:]
fig,axes=plt.subplots(1,3,figsize=(14,3.7));frame.queue.value_counts().plot.bar(ax=axes[0],color="#176b66");axes[0].set_title("Routing classes")
axes[1].barh(names[top],counts[top],color="#d38b45");axes[1].set_title("Frequent tokens")
frame.assign(words=frame.text.str.split().str.len()).boxplot(column="words",by="queue",ax=axes[2]);axes[2].set_title("Message lengths");plt.suptitle("");plt.tight_layout();plt.show()

**Takeaway:** Balanced queues make the class priors comparable.

**Takeaway:** Generic words are common but weak because they occur in every queue.

**Takeaway:** Similar document lengths prevent length from becoming an accidental shortcut.

# 5. Prepare

We create the evaluation boundary before learning any transformation or model choice. Training-only operations remain inside the fitted workflow so held-out rows cannot influence them.

Any learned scaling, encoding, imputation, or resampling belongs after
the split—normally inside a pipeline. Doing it on all rows would let
held-out information influence training and would make evaluation too
optimistic.

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(frame.text,frame.queue,test_size=.25,stratify=frame.queue,random_state=SEED)
model=make_pipeline(CountVectorizer(),MultinomialNB(alpha=1));print("Train/test:",X_train.shape,X_test.shape)

# 6. Train

        Training turns the assumptions behind naive bayes for text into an explicit fitted rule. The printed intermediate values expose what was learned rather than treating fit() as magic.

        Conceptually, fitting follows these steps:

        1. Estimate class priors from training documents.
2. Count tokens separately inside each class.
3. Add alpha pseudo-counts to avoid zero likelihoods.
4. Sum class-prior and token log likelihoods for each message.

In [ ]:
model.fit(X_train,y_train);nb=model.named_steps["multinomialnb"];names=model.named_steps["countvectorizer"].get_feature_names_out()
for c,row in zip(nb.classes_,nb.feature_log_prob_): print(c,"top words:",names[np.argsort(row)[-5:]].tolist())

# 7. Evaluate

We compare against a deliberately simple baseline, report a task-appropriate metric, and save three diagnostic views that explain where the score came from.

We report a baseline as well as the model and save the central figure as
`key_figure.png`. A model earns attention only by beating a simple,
honestly evaluated alternative.

In [ ]:
pred=model.predict(X_test);base=np.repeat(pd.Series(y_train).mode()[0],len(y_test));print(f"NB accuracy={accuracy_score(y_test,pred):.3f}; baseline={accuracy_score(y_test,base):.3f}")
proba=model.predict_proba(X_test);fig,axes=plt.subplots(1,3,figsize=(14,3.7));ConfusionMatrixDisplay.from_predictions(y_test,pred,ax=axes[0],colorbar=False);axes[0].set_title("Held-out routing")
for j,c in enumerate(nb.classes_):
    ids=np.argsort(nb.feature_log_prob_[j])[-5:];axes[1].plot(nb.feature_log_prob_[j,ids],names[ids],"o",label=c)
axes[1].set_title("Strong token evidence");axes[1].legend(fontsize=7);axes[2].hist(proba.max(axis=1),bins=10,color="#176b66");axes[2].set(title="Confidence",xlabel="largest probability");plt.tight_layout();plt.savefig("key_figure.png",dpi=160,bbox_inches="tight");plt.show()

**Takeaway:** The model improves on the simple baseline where its assumptions match the data; the diagnostics show that the aggregate score alone is not enough.

# 8. Break it

We now violate the important condition directly: Token evidence is treated as conditionally independent, so order-dependent phrases and repeated correlated clues can fool confidence. The resulting number makes the cost of that violation visible.

The following experiment deliberately violates an assumption or removes
a protection. Compare the printed damage with the healthy evaluation;
the failure is part of the lesson, not an accidental exception.

In [ ]:
phrase_x=np.array((["not bad","good"]*60)+(["not good","bad"]*60));phrase_y=np.array(([1,1]*60)+([0,0]*60));order=np.random.default_rng(SEED).permutation(len(phrase_x));phrase_x,phrase_y=phrase_x[order],phrase_y[order]
xa,xb,ya,yb=train_test_split(phrase_x,phrase_y,test_size=.3,stratify=phrase_y,random_state=SEED)
uni=make_pipeline(CountVectorizer(),MultinomialNB()).fit(xa,ya);bi=make_pipeline(CountVectorizer(ngram_range=(1,2)),MultinomialNB()).fit(xa,ya)
print(f"Unigram accuracy={uni.score(xb,yb):.3f}; phrase-feature accuracy={bi.score(xb,yb):.3f}")

**Use this when…** sparse word counts carry the signal and you need a fast, inspectable baseline.

        **Don't use this when…** order, negation, long context, or duplicated correlated evidence determine meaning.

        ## Try this

        1. Sweep alpha from .01 to 100 and inspect rare-word confidence.
2. Add word bigrams and retry negation.
3. Add character n-grams and misspell 'refund'.